# 🔬 End-to-End Pipeline Diagnostics & Flow Verification

สมุดสำหรับตรวจสอบและดีบักระบบ Edge AI ทุก Flow อย่างละเอียด:
- **Flow 1:** Preprocessing & Channel Order (HWC vs CHW)
- **Flow 2:** FPGA Golden Vector Self-Test (`test_input_int8.bin` vs `test_output_int8.bin`)
- **Flow 3:** FPGA Output Feature Map & 32-Channel Activation Heatmap
- **Flow 4:** Classifier Logits, Top-5 Predictions & Full 31-Class Probabilities Bar Chart

In [ ]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

from src.pynq_driver import PlantDiseaseAccelerator
from src.preprocessing import preprocess_image
from src.postprocessing import process_output, PLANT_DISEASE_CLASSES

print("✅ โหลดโมดูลสำหรับ Diagnostic เรียบร้อย")

## 🛠️ Step 0: Initialize Accelerator (Bitstream V2)

In [ ]:
accel = PlantDiseaseAccelerator(bitstream_path="../overlays/plant_disease.bit", weights_dir="../weights")
print(f"🚀 Accelerator Status: {accel.mode.upper()}")
print(f"📌 Bitstream Path: {accel.bitstream_path}")

## 🧪 FLOW 2 TEST: FPGA Hardware Golden Vector Self-Test
ทดสอบส่งไฟล์ทดสอบมาตรฐาน (`test_input_int8.bin`) เข้าชิป FPGA ตรงๆ และเทียบกับ `test_output_int8.bin` ของคนที่ 1

In [ ]:
in_path = "../test_vectors/test_input_int8.bin"
out_path = "../test_vectors/test_output_int8.bin"

if os.path.exists(in_path) and os.path.exists(out_path):
    print("⏳ กำลังรัน Golden Vector บน FPGA...")
    res = accel.test_golden_vector(in_path, out_path)
    print("=========================================")
    print(f"⏱️ FPGA Execution Latency: {res['hw_time_ms']} ms")
    print(f"📊 Mean Squared Error (MSE): {res['mse']:.6f}")
    print(f"🎯 Match Percentage: {res['match_pct']}%")
    print(f"🏆 Verification Result: {'[PASSED] ✅' if res['passed'] else '[FAILED] ❌'}")
    print("=========================================")
else:
    print(f"⚠️ ไม่พบไฟล์ test_vectors ใน {in_path}")

## 📷 FLOW 1 TEST: Camera Frame & Preprocessing Data Layout (CHW)
ตรวจสอบการแยกช่องสี R, G, B และการแปลงภาพเข้าสู่ Channel-First `(3, 224, 224)`

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
ret, raw_frame = cap.read()
cap.release()

if not ret:
    print("⚠️ ไม่สามารถเปิดกล้องได้ - ใช้ภาพสังเคราะห์แทน")
    raw_frame = np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8)

# Preprocess to CHW INT8
chw_input = preprocess_image(raw_frame, to_chw=True)
hwc_input = preprocess_image(raw_frame, to_chw=False)

print(f"📐 Raw Frame Shape: {raw_frame.shape}")
print(f"📐 CHW Buffer Shape: {chw_input.shape} (Dtype: {chw_input.dtype})")
print(f"📊 Value Range: Min={chw_input.min()}, Max={chw_input.max()}, Mean={chw_input.mean():.2f}")

# พล็อตภาพ Original และแยกช่อง R, G, B
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
axs[0].imshow(cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB))
axs[0].set_title("Original Image")
axs[0].axis('off')

axs[1].imshow(chw_input[0], cmap='Reds')
axs[1].set_title("Channel 0: Red (INT8)")
axs[1].axis('off')

axs[2].imshow(chw_input[1], cmap='Greens')
axs[2].set_title("Channel 1: Green (INT8)")
axs[2].axis('off')

axs[3].imshow(chw_input[2], cmap='Blues')
axs[3].set_title("Channel 2: Blue (INT8)")
axs[3].axis('off')

plt.tight_layout()
plt.show()

## 🧠 FLOW 3 & 4 TEST: FPGA Feature Map & Top-5 Disease Classification
ส่งภาพจากกล้องเข้า FPGA $\rightarrow$ อ่าน Feature Map 32 Channels $\rightarrow$ สรุปผลความน่าจะเป็นทั้ง 31 โรคพืช

In [ ]:
# ส่งภาพเข้า FPGA
accel.write_input(chw_input)
t0 = time.time()
accel.run_inference()
hw_latency = (time.time() - t0) * 1000.0

# อ่าน Raw Logits
logits = accel.read_output()

# คำนวณ Softmax Probabilities
e_x = np.exp(logits - np.max(logits))
probs = e_x / np.sum(e_x)

# หา Top-5 Classes
top5_idx = np.argsort(probs)[::-1][:5]

print("===========================================================")
print(f"⚡ FPGA Inference Latency: {hw_latency:.2f} ms")
print("🏆 TOP-5 DETECTED PLANT DISEASES:")
print("-----------------------------------------------------------")
for rank, idx in enumerate(top5_idx, 1):
    c_info = PLANT_DISEASE_CLASSES[idx]
    print(f"#{rank} [{probs[idx]*100:5.1f}%]  {c_info['en']}  ({c_info['th']})")
print("===========================================================")

# พล็อต Bar Chart ของความน่าจะเป็นทั้ง 31 คลาส
plt.figure(figsize=(14, 6))
class_names_short = [c['en'].split('_')[0] + ' ' + '_'.join(c['en'].split('_')[1:]) for c in PLANT_DISEASE_CLASSES]
bars = plt.bar(range(len(probs)), probs * 100, color='forestgreen')
bars[top5_idx[0]].set_color('crimson') # ไฮไลต์อันดับ 1

plt.xticks(range(len(probs)), class_names_short, rotation=90, fontsize=8)
plt.ylabel("Confidence (%)")
plt.title(f"31-Class Disease Probability Distribution (Top: {PLANT_DISEASE_CLASSES[top5_idx[0]]['en']})")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 🧹 Cleanup Memory

In [ ]:
accel.cleanup()
print("🧹 เคลียร์ Memory Buffers เรียบร้อย")